# Fine-tuning Llama 3 8B with QLoRA (Unsloth) on Google Colab

This notebook fine-tunes `unsloth/llama-3-8b-bnb-4bit` on a custom instruction dataset using QLoRA, on a free Colab T4 GPU.

**Runtime > Change runtime type > T4 GPU** before running.

## 1. Install dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl==0.9.6 peft==0.12.0 accelerate==0.33.0 bitsandbytes==0.43.3

## 2. Load base model in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect (bf16 on A100, fp16 on T4)
    load_in_4bit=True,
)

## 3. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

## 4. Load and format the dataset

Run `python data_prep.py` locally first, or upload `data/processed/train.jsonl` directly to Colab.

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("json", data_files="data/processed/train.jsonl", split="train")
val_dataset = load_dataset("json", data_files="data/processed/val.jsonl", split="train")

print(train_dataset[0]["text"][:500])

## 5. Train with TRL's SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

## 6. Save the LoRA adapter

In [ ]:
model.save_pretrained("lora_adapters/llama3-8b-custom")
tokenizer.save_pretrained("lora_adapters/llama3-8b-custom")

# Optional: push to Hugging Face Hub
# model.push_to_hub("your-username/llama3-8b-custom-lora", token="hf_...")
# tokenizer.push_to_hub("your-username/llama3-8b-custom-lora", token="hf_...")

## 7. Quick inference test

In [ ]:
FastLanguageModel.for_inference(model)

prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Explain what QLoRA is in simple terms.

### Response:
"""

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])